## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
import os
#os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")
os.environ["WANDB_MODE"] = "offline"

In [ ]:
"""!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None"""

### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# HEAT MAP KAI SCATTER PLOT ΓΙΑ ΤΑ Decoder model and CNNs.

In [ ]:
# ================================================================
# 0. INSTALLS
# ================================================================
#!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes

!pip install -q umap-learn transformers peft

import os, math, time, random, string
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
import umap

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    GPTNeoForCausalLM,
    BitsAndBytesConfig,
)
from peft import get_peft_model, LoraConfig, TaskType

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
import nltk
import pickle

# NLTK stuff
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

LABEL_NAMES = ['Negative', 'Neutral', 'Positive']   # 0,1,2

In [ ]:
# Εισαγωγή του Access Token απο την Hugging face.
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
from huggingface_hub import login

if not os.environ.get("HF_TOKEN"):
    raise ValueError("Set the HF_TOKEN environment variable before loading gated Hugging Face models.")

login(token=os.environ["HF_TOKEN"])

In [ ]:
# ================================================================
# 1. DATA LOADING & PREPROCESSING  (κοινό για όλα τα μοντέλα)
# ================================================================
DATA_PATH = "data/external/sentences_dataset_45269.csv"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    filtered_words = [w for w in words if w not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in filtered_words]
    return " ".join(lemmatized_words)

df = pd.read_csv(DATA_PATH)


"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(30))

df["sentence"] = df["sentence"].apply(clean_text)

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(30))"""


# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.
# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)


# ίδιο split όπως στα περισσότερα scripts (30% temp, 50-50 val/test)
train_data, temp_data = train_test_split(
    df, test_size=0.3, stratify=df["polarity"], random_state=SEED
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, stratify=temp_data["polarity"], random_state=SEED
)

print("Train:", train_data.shape, "Val:", val_data.shape, "Test:", test_data.shape)

test_texts = test_data["sentence"].tolist()
test_labels = test_data["polarity"].to_numpy()


# μικρό helper: batch-ify λίστα
def batch_iter(lst, batch_size):
    for i in range(0, len(lst), batch_size):
        yield lst[i:i+batch_size]

In [ ]:
# ================================================================
# 2. MODELS CONFIG (paths)
# ================================================================
BASE_DIR = "external_materials/model_weights"

paths = {
    "DeepSeek": {
        "state_path": f"{BASE_DIR}/DeepSeek/saved_model/saved_deepSeek_state_dict.pth",
        "tok_path":   f"{BASE_DIR}/DeepSeek/saved_tokenizer",
        "hf_name":    "deepseek-ai/deepseek-llm-7b-base",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    },
    "LLaMA_3": {
        "state_path": f"{BASE_DIR}/LLaMA_3/saved_model/saved_llama3_state_dict.pth",
        "tok_path":   f"{BASE_DIR}/LLaMA_3/saved_tokenizer",
        "hf_name":    "meta-llama/Llama-3.1-8B",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    },
}


# Το κάτω τμήμα θα μπει αργότερα στο λεξικό paths.

"""
    "GEMMA": {
        "state_path": f"{BASE_DIR}/GEMMA/saved_model/saved_gemma_state_dict.pth",
        "tok_path":   f"{BASE_DIR}/GEMMA/saved_tokenizer",
        "hf_name":    "google/gemma-7b",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    },
    "GPT_2": {
        "model_dir":  f"{BASE_DIR}/GPT_2/saved_model",
        "tok_path":   f"{BASE_DIR}/GPT_2/saved_tokenizer",

    },
    "GPT_Neo": {
        "model_dir":  f"{BASE_DIR}/GPT_Neo/gpt_neo_125m/saved_model",
        "tok_path":   f"{BASE_DIR}/GPT_Neo/gpt_neo_125m/saved_tokenizer",

    },
    "CNN": {
        "model_path": f"{BASE_DIR}/CNN/saved_model/sentiment_cnn.pth",
        "tok_path":   f"{BASE_DIR}/CNN/saved_tokenizer/tokenizer.pkl",
        "max_vocab":  20000,
        "max_len":    256,
    },
    "CNN_V2": {
        "model_path": f"{BASE_DIR}/CNN_V2/saved_model/sentiment_cnn.pth",
        "tok_path":   f"{BASE_DIR}/CNN_V2/saved_tokenizer/tokenizer.pkl",
        "max_vocab":  20000,
        "max_len":    256,
    },
"""

In [ ]:
# ================================================================
# 3. HELPERS ΓΙΑ LORA SEQ_CLS ΜΟΝΤΕΛΑ (DeepSeek / Falcon / GEMMA / LLaMA_3)
# ================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

class LoRASeqClassifier(nn.Module):
    """
    Generic wrapper: base_model (SeqClassification) + δική μου linear classifier
    και δυνατότητα να γυρίσει embeddings (CLS).
    """
    def __init__(self, base_model, num_labels=3):
        super().__init__()
        self.base_model = base_model
        # το ίδιο dtype με το training: fp16
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]  # [B, L, H]

        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)

        pooled = pooled.to(self.classifier.weight.dtype)
        logits = self.classifier(pooled)

        if return_hidden:
            return logits, pooled
        return logits


def load_lora_seqcls_model(name):
    cfg = paths[name]
    tok = AutoTokenizer.from_pretrained(cfg["tok_path"], fix_mistral_regex=True)
    tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.pad_token

    base = AutoModelForSequenceClassification.from_pretrained(
        cfg["hf_name"],
        quantization_config=bnb_config,
        num_labels=3,
    )
    base.config.pad_token_id = tok.pad_token_id

    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=cfg["target_modules"],
    )
    base = get_peft_model(base, lora_cfg)

    model = LoRASeqClassifier(base, num_labels=3)
    state = torch.load(cfg["state_path"], map_location="cpu")
    model.load_state_dict(state, strict=False)
    model.to(device)
    model.eval()
    return tok, model


def eval_lora_seqcls(name, texts, labels, max_samples=None, batch_size=16):
    print(f"\n=== {name}: running inference on test set ===")
    tok, model = load_lora_seqcls_model(name)

    if max_samples is not None:
        texts = texts[:max_samples]
        labels = labels[:max_samples]

    all_preds = []
    all_embs = []

    with torch.no_grad():
        for batch_texts in batch_iter(texts, batch_size):
            enc = tok(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            ).to(device)

            logits, embeddings = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                return_hidden=True,
            )

            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)
            all_embs.append(embeddings.detach().float().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_embs = np.vstack(all_embs)
    cm = confusion_matrix(labels[:len(all_preds)], all_preds, labels=[0,1,2])
    return cm, all_embs, all_preds

In [ ]:
# ================================================================
# 4. CNN MODEL (για embeddings & predictions)
# ================================================================
class CNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x, return_features=False):
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.mean(x, dim=2)   # [B, 128]
        x = F.relu(self.fc1(x))    # [B, 64]
        features = x
        logits = self.fc2(x)
        if return_features:
            return logits, features
        return logits


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def load_cnn_model_and_tokenizer():
    cfg = paths["CNN"]
    with open(cfg["tok_path"], "rb") as f:
        keras_tok = pickle.load(f)

    model = CNNModel(cfg["max_vocab"], embed_dim=128, num_classes=3)
    state = torch.load(cfg["model_path"], map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()

    def text_pipeline(texts):
        seqs = keras_tok.texts_to_sequences(texts)
        padded = pad_sequences(seqs, maxlen=cfg["max_len"], padding="post")
        return torch.tensor(padded, dtype=torch.long)

    return model, text_pipeline


def load_cnn_model_and_tokenizer_for_CNN_V2():
    cfg = paths["CNN_V2"]
    with open(cfg["tok_path"], "rb") as f:
        keras_tok = pickle.load(f)

    model = CNNModel(cfg["max_vocab"], embed_dim=128, num_classes=3)
    state = torch.load(cfg["model_path"], map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()

    def text_pipeline(texts):
        seqs = keras_tok.texts_to_sequences(texts)
        padded = pad_sequences(seqs, maxlen=cfg["max_len"], padding="post")
        return torch.tensor(padded, dtype=torch.long)

    return model, text_pipeline


def eval_cnn(texts, labels, max_samples=None, batch_size=16):
    print("\n=== CNN: running inference on test set ===")
    model, text_pipeline = load_cnn_model_and_tokenizer()
    if max_samples is not None:
        texts = texts[:max_samples]
        labels = labels[:max_samples]

    all_preds = []
    all_embs = []

    with torch.no_grad():
        for batch_texts in batch_iter(texts, batch_size):
            x = text_pipeline(batch_texts).to(device)
            logits, features = model(x, return_features=True)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)
            all_embs.append(features.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_embs = np.vstack(all_embs)
    cm = confusion_matrix(labels[:len(all_preds)], all_preds, labels=[0,1,2])
    return cm, all_embs, all_preds


def eval_cnn_for_CNN_V2(texts, labels, max_samples=None, batch_size=16):
    print("\n=== CNN: running inference on test set ===")
    model, text_pipeline = load_cnn_model_and_tokenizer_for_CNN_V2()
    if max_samples is not None:
        texts = texts[:max_samples]
        labels = labels[:max_samples]

    all_preds = []
    all_embs = []

    with torch.no_grad():
        for batch_texts in batch_iter(texts, batch_size):
            x = text_pipeline(batch_texts).to(device)
            logits, features = model(x, return_features=True)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)
            all_embs.append(features.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_embs = np.vstack(all_embs)
    cm = confusion_matrix(labels[:len(all_preds)], all_preds, labels=[0,1,2])
    return cm, all_embs, all_preds

In [ ]:
# ================================================================
# 5. GPT-2 & GPT-Neo (CAUSAL LM με generate για labels + embeddings)
# ================================================================
label_map = {0: "negative", 1: "neutral", 2: "positive"}
inv_label_map = {v: k for k, v in label_map.items()}


def load_gpt2_model():
    cfg = paths["GPT_2"]
    tok = GPT2Tokenizer.from_pretrained(cfg["tok_path"])
    tok.pad_token = tok.eos_token
    model = GPT2LMHeadModel.from_pretrained(cfg["model_dir"])
    model.config.pad_token_id = tok.pad_token_id
    model.to(device)
    model.eval()
    return tok, model


def load_gptneo_model():
    cfg = paths["GPT_Neo"]
    tok = GPT2Tokenizer.from_pretrained(cfg["tok_path"])
    tok.pad_token = tok.eos_token
    model = GPTNeoForCausalLM.from_pretrained(cfg["model_dir"])
    model.config.pad_token_id = tok.pad_token_id
    model.to(device)
    model.eval()
    return tok, model


def gpt_eval_and_embeddings(model_name, texts, labels, max_samples=None, batch_size=16):
    print(f"\n=== {model_name}: running generate() on test set ===")

    if model_name == "GPT_2":
        tok, model = load_gpt2_model()
    else:
        tok, model = load_gptneo_model()

    if max_samples is not None:
        texts = texts[:max_samples]
        labels = labels[:max_samples]

    valid_tokens = [tok.convert_tokens_to_ids(t) for t in ["negative", "neutral", "positive"]]

    from transformers import LogitsProcessorList, LogitsProcessor

    class SentimentLogitsProcessor(LogitsProcessor):
        def __call__(self, input_ids, scores):
            # κρατά μόνο τα 3 tokens
            mask = torch.full_like(scores, float("-inf"))
            mask[:, valid_tokens] = scores[:, valid_tokens]
            return mask

    logits_processor = LogitsProcessorList([SentimentLogitsProcessor()])

    all_preds = []
    all_embs = []

    with torch.no_grad():
        for batch_texts in batch_iter(texts, batch_size):
            prompts = [
                f"Classify the polarity sentiment of the following sentence.\n\n{text}\nSentiment:"
                for text in batch_texts
            ]
            enc = tok(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(device)

            # embeddings: last hidden state mean-pooled
            outputs = model(
                **enc,
                output_hidden_states=True
            )
            last_hidden = outputs.hidden_states[-1]      # [B, L, H]
            emb = last_hidden.mean(dim=1).cpu().numpy()  # [B, H]
            all_embs.append(emb)

            gen_ids = model.generate(
                **enc,
                max_new_tokens=1,
                logits_processor=logits_processor,
                pad_token_id=tok.pad_token_id,
            )

            # μόνο τα τελευταία tokens
            new_tokens = gen_ids[:, enc["input_ids"].shape[1]:]
            texts_out = tok.batch_decode(new_tokens, skip_special_tokens=True)
            preds = []
            for t in texts_out:
                t = t.strip().lower()
                if "negative" in t:
                    preds.append(inv_label_map["negative"])
                elif "positive" in t:
                    preds.append(inv_label_map["positive"])
                elif "neutral" in t:
                    preds.append(inv_label_map["neutral"])
                else:
                    preds.append(inv_label_map["neutral"])  # fallback
            all_preds.append(np.array(preds, dtype=int))

    all_preds = np.concatenate(all_preds)
    all_embs = np.vstack(all_embs)
    cm = confusion_matrix(labels[:len(all_preds)], all_preds, labels=[0,1,2])
    return cm, all_embs, all_preds

In [ ]:
def save_classification_report(model_name, y_true, y_pred, class_names, save_path):
    """
    Δημιουργεί PDF image με classification report για συγκεκριμένο μοντέλο.
    """

    # Παράγουμε report ως string
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3
    )

    # Δημιουργούμε εικόνα
    fig, ax = plt.subplots(figsize=(8.5, 6))
    ax.axis("off")

    # Τοποθετούμε το κείμενο στο plot
    ax.text(
        0.01, 0.99,
        f"Classification Report — {model_name}\n\n{report}",
        fontsize=10,
        va="top",
        family="monospace"
    )

    fig.tight_layout()

    # Αποθήκευση PDF
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# ================================================================
# 6. ΤΡΕΧΟΥΜΕ ΟΛΑ ΤΑ ΜΟΝΤΕΛΑ & ΜΑΖΕΥΟΥΜΕ ΑΠΟΤΕΛΕΣΜΑΤΑ
# ================================================================
conf_mats = {}
embeddings = {}
predictions = {}

MAX_SAMPLES_UMAP = None  # ή βάζω π.χ. 2000 για πιο ελαφρύ

collection = int(input("Επίλεξε ομάδα μοντέλων: [1: DeepSeek / Falcon / GEMMA / LLaMA], [2: CNNs], [3: GPTs]"))

if collection == 1:
    # DeepSeek / Falcon / GEMMA / LLaMA
    for name in ["LLaMA_3", "DeepSeek"]: # , "Falcon", "GEMMA"
        cm, embs, preds = eval_lora_seqcls(name, test_texts, test_labels, max_samples=MAX_SAMPLES_UMAP)
        conf_mats[name] = cm
        embeddings[name] = embs
        predictions[name] = preds

        report_path = f"outputs/figures/fig_comparable/{name}_classification_report.pdf"
        save_classification_report(name, test_labels, preds, LABEL_NAMES, report_path)
        print(f"Saved classification report for {name} -> {report_path}")

if collection == 2:
    # CNN
    cm, embs, preds = eval_cnn(test_texts, test_labels, max_samples=MAX_SAMPLES_UMAP)
    conf_mats["CNN"] = cm
    embeddings["CNN"] = embs
    predictions["CNN"] = preds

    report_path = f"outputs/figures/fig_comparable/CNN_classification_report.pdf"
    save_classification_report("CNN", test_labels, preds, LABEL_NAMES, report_path)
    print(f"Saved classification report for CNN -> {report_path}")

    # CNN_V2
    cm, embs, preds = eval_cnn_for_CNN_V2(test_texts, test_labels, max_samples=MAX_SAMPLES_UMAP)
    conf_mats["CNN_V2"] = cm
    embeddings["CNN_V2"] = embs
    predictions["CNN_V2"] = preds

    report_path = f"outputs/figures/fig_comparable/CNN_V2_classification_report.pdf"
    save_classification_report("CNN_V2", test_labels, preds, LABEL_NAMES, report_path)
    print(f"Saved classification report for CNN_V2 -> {report_path}")

if collection == 3:
    # GPT-2 & GPT-Neo
    cm, embs, preds = gpt_eval_and_embeddings("GPT_2", test_texts, test_labels, max_samples=MAX_SAMPLES_UMAP)
    conf_mats["GPT_2"] = cm
    embeddings["GPT_2"] = embs
    predictions["GPT_2"] = preds

    report_path = f"outputs/figures/fig_comparable/GPT_2_classification_report.pdf"
    save_classification_report("GPT_2", test_labels, preds, LABEL_NAMES, report_path)
    print(f"Saved classification report for GPT_2 -> {report_path}")

    cm, embs, preds = gpt_eval_and_embeddings("GPT_Neo", test_texts, test_labels, max_samples=MAX_SAMPLES_UMAP)
    conf_mats["GPT_Neo"] = cm
    embeddings["GPT_Neo"] = embs
    predictions["GPT_Neo"] = preds

    report_path = f"outputs/figures/fig_comparable/GPT_Neo_classification_report.pdf"
    save_classification_report("GPT_Neo", test_labels, preds, LABEL_NAMES, report_path)
    print(f"Saved classification report for GPT_Neo -> {report_path}")

In [ ]:
# ================================================================
# 7. HEATMAPS (CONFUSION MATRICES) – 2 ανά σειρά
# ================================================================
sns.set(style="whitegrid", font_scale=1.0)

if collection == 1:
    model_order = ["LLaMA_3", "DeepSeek"] # , "Falcon", "GEMMA"

if collection == 2:
    model_order = ["CNN", "CNN_V2"]

if collection == 3:
    model_order = ["GPT_2", "GPT_Neo"]

n_models = len(model_order)
cols = 2
rows = math.ceil(n_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(9, 4 * rows))
axes = axes.flatten()

for i, name in enumerate(model_order):
    ax = axes[i]
    cm = conf_mats[name]
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        cbar=False, ax=ax
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted Labels")
    ax.set_ylabel("True Labels")

for j in range(n_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
if collection == 1:
    fig.savefig(
        f'outputs/figures/fig_comparable/decoder_only_models_heat_maps.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )
if collection == 2:
    fig.savefig(
        f'outputs/figures/fig_comparable/CNNs_heat_maps.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )
if collection == 3:
    fig.savefig(
        f'outputs/figures/fig_comparable/GPTs_heat_maps.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )

In [ ]:
# ================================================================
# 8. UMAP SCATTER PLOTS – 2 ανά σειρά (seaborn)
#    Χρωματίζουμε με βάση το TRUE label
# ================================================================
fig, axes = plt.subplots(rows, cols, figsize=(9, 4 * rows))
axes = axes.flatten()

for i, name in enumerate(model_order):
    ax = axes[i]
    embs = embeddings[name]

    # UMAP σε 2D
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=SEED
    )
    embs_2d = reducer.fit_transform(embs)

    # labels για τα ίδια δείγματα
    y = test_labels[:embs_2d.shape[0]]

    # Φτιάχνουμε DataFrame για seaborn
    df_plot = pd.DataFrame({
        "UMAP1": embs_2d[:, 0],
        "UMAP2": embs_2d[:, 1],
        "label_id": y,
        "label": [LABEL_NAMES[int(l)] for l in y]
    })

    palette = {
        "Negative": "tab:red",
        "Neutral": "tab:blue",
        "Positive": "tab:green"
    }

    sns.scatterplot(
        data=df_plot,
        x="UMAP1",
        y="UMAP2",
        hue="label",          # χρώμα ανά κλάση (TRUE label)
        s=20,
        palette=palette,
        alpha=0.7,
        ax=ax
    )

    ax.set_title(f"{name} Embeddings")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(title="Sentiment", fontsize=10)

# Κρύβουμε τυχόν άδεια subplots
for j in range(n_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
if collection == 1:
    fig.savefig(
        f'outputs/figures/fig_comparable/decoder_only_models_umap_scatter.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )
if collection == 2:
    fig.savefig(
        f'outputs/figures/fig_comparable/CNNs_umap_scatter.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )
if collection == 3:
    fig.savefig(
        f'outputs/figures/fig_comparable/GPTs_umap_scatter.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )

Master Results File. Προσθήκη και των προβλέψεων των DeepSeek και LLaMA 3.

In [ ]:
# ================================================================
# 9. UPDATE MASTER RESULTS CSV WITH DeepSeek & LLaMA-3 PREDICTIONS
# ================================================================

MASTER_PATH = (
    "data/processed/Master_results.csv"
)

# Φόρτωση Master CSV
master_df = pd.read_csv(MASTER_PATH)
print("Master CSV loaded. Shape:", master_df.shape)

# ------------------------------------------------
# Έλεγχος ότι υπάρχουν οι προβλέψεις
# ------------------------------------------------
assert "DeepSeek" in predictions, " DeepSeek predictions not found!"
assert "LLaMA_3" in predictions, " LLaMA_3 predictions not found!"

deepseek_preds = predictions["DeepSeek"]
llama3_preds   = predictions["LLaMA_3"]

# ------------------------------------------------
# Κρίσιμος έλεγχος ευθυγράμμισης
# ------------------------------------------------
assert len(master_df) == len(deepseek_preds) == len(llama3_preds), (
    " Mismatch between Master CSV rows and DeepSeek / LLaMA-3 predictions!"
)

# ------------------------------------------------
# Προσθήκη στηλών
# ------------------------------------------------
master_df["DeepSeek_pred"] = deepseek_preds
master_df["LLaMA3_pred"]   = llama3_preds

# ------------------------------------------------
# Αποθήκευση
# ------------------------------------------------
master_df.to_csv(MASTER_PATH, index=False, encoding="utf-8")

print(f" Master_results.csv updated with DeepSeek & LLaMA-3 predictions -> {MASTER_PATH}")


Επειδή το dataset περιέχει και προτάσεις που δεν είναι citations, λόγω του ότι θα χρησιμοποιηθεί σε άλλες εργασίες και αναγνώριση αν μια πρόταση είναι citation η όχι, η στήλη citation μετονομάστηκε σε sentence στον παρακάτω κώδικα.

In [ ]:
# Μετονομασία στήλης
master_df.rename(columns={"citation": "sentence"}, inplace=True)

master_df.to_csv(MASTER_PATH, index=False, encoding="utf-8")
print("Η στήλη 'citation' μετονομάστηκε σε 'sentence' και το CSV αποθηκεύτηκε ξανά.")

END